In [ ]:
import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix
from sklearn.utils import class_weight

from lightgbm import LGBMClassifier
import seaborn as sns
import matplotlib.pyplot as plt

In this phase, the proposed methodology was compared with recent research work to identify potential gaps. Based on this analysis, improvements were made to enhance model performance, handle limitations, and make the approach more robust for IoT attack classification.

In [ ]:
df = pd.read_csv("/kaggle/input/datasets/malaikatanveer123/feature-engineered-iot-dataset/feature_engineered_iot_dataset.csv")

df.head()

In [ ]:
X = df.drop(['type','label'], axis=1)
y = df['type']

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

In [ ]:
class_weights = class_weight.compute_class_weight(
    class_weight='balanced',
    classes=np.unique(y_train),
    y=y_train
)

class_weights_dict = dict(zip(np.unique(y_train), class_weights))

print(class_weights_dict)

One major limitation identified in the initial methodology was class imbalance, where some attack categories had significantly fewer samples than others. This imbalance negatively affects model performance, especially for minority classes.

To address this issue, class weights were computed and applied during model training. This ensures that the model gives more importance to underrepresented classes, improving recall and overall classification performance for those attack types.

In [ ]:
lgb = LGBMClassifier(class_weight=class_weights_dict)

lgb.fit(X_train, y_train)

y_pred = lgb.predict(X_test)

The LightGBM model was retrained using class weights to improve its ability to generalize across all attack categories. This enhancement helps the model better learn patterns from imbalanced data and reduces bias toward majority classes.

In [ ]:
print("Improved LightGBM Accuracy:", accuracy_score(y_test, y_pred))
print(classification_report(y_test, y_pred))

In [ ]:
cm = confusion_matrix(y_test, y_pred)

plt.figure(figsize=(8,6))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues')
plt.title("Confusion Matrix - Improved Model")
plt.show()

In [ ]:
importance = pd.Series(lgb.feature_importances_, index=X.columns)
importance = importance.sort_values(ascending=False)

importance.head(15)

Another limitation observed was the presence of less important features, which could introduce noise and reduce model efficiency.

To overcome this, feature importance analysis was used to select the top 15 most important features. By training the model on these selected features, the dimensionality of the dataset was reduced, leading to improved performance, faster training, and better generalization.

In [ ]:
top_features = importance.head(15).index

X_top = df[top_features]
y = df['type']

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X_top, y, test_size=0.2, random_state=42
)

lgb2 = LGBMClassifier(class_weight=class_weights_dict)

lgb2.fit(X_train, y_train)

y_pred2 = lgb2.predict(X_test)

After applying class imbalance handling and feature selection, the model was retrained and evaluated again. The improved model showed better performance, particularly in detecting minority attack classes, while maintaining high overall accuracy.

In [ ]:
print("Final Model Accuracy:", accuracy_score(y_test, y_pred2))
print(classification_report(y_test, y_pred2))

The IoT Attack Risk Indicator was further improved using the optimized model. This allows the system to not only detect whether traffic is malicious but also predict the specific type of attack more accurately, providing deeper insights into network threats.

In [ ]:
df['attack_risk'] = lgb2.predict(X_top)

Most existing research focuses on either binary classification or does not effectively handle class imbalance and feature redundancy. In contrast, this work addresses these gaps by:

- Performing multi-class classification of IoT attacks  
- Applying feature engineering and feature selection  
- Handling class imbalance using class weights  
- Providing an interpretable IoT Attack Risk Indicator  

These improvements make the proposed methodology more practical and effective for real-world IoT security applications.

In [ ]:
df.to_csv("final_improved_iot_dataset.csv", index=False)

The improved methodology successfully enhances the performance and robustness of the IoT attack classification system. By addressing key limitations such as class imbalance and irrelevant features, the model becomes more accurate and reliable in detecting various types of cyber attacks.